In [15]:
import os
import re
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")

CHANNEL_URLS = [
    "https://www.youtube.com/@UNCTADOnline/videos",
]

OUTPUT_PATH = "youtube_videos.xlsx"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

SLEEP_SECONDS = 0.1
MAX_PAGES = 50  # None이면 끝까지 수집

channel_url = CHANNEL_URLS[0]
html = requests.get(channel_url, headers=HEADERS, timeout=30).text
match = re.search(r'"externalId":"(UC[^"]+)"', html)

if not match:
    raise ValueError("채널 ID를 찾지 못했습니다.")

channel_id = match.group(1)

print("channel_url:", channel_url)
print("channel_id:", channel_id)

channel_url: https://www.youtube.com/@UNCTADOnline/videos
channel_id: UCaehHbVEHmss6hOYt-mjnWg


In [16]:
url = "https://www.googleapis.com/youtube/v3/channels"

params = {
    "part": "snippet,contentDetails",
    "id": channel_id,
    "key": API_KEY,
}

res = requests.get(url, params=params, timeout=30)
res.raise_for_status()

data = res.json()

item = data["items"][0]

channel_name = item["snippet"]["title"]
uploads_playlist_id = item["contentDetails"]["relatedPlaylists"]["uploads"]

print("channel_name:", channel_name)
print("uploads_playlist_id:", uploads_playlist_id)

channel_name: UN Trade and Development
uploads_playlist_id: UUaehHbVEHmss6hOYt-mjnWg


In [17]:
all_videos = []

next_page_token = None
page_count = 0

while True:
    params = {
        "part": "snippet,contentDetails",
        "playlistId": uploads_playlist_id,
        "maxResults": 50,
        "key": API_KEY,
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    res = requests.get(
        "https://www.googleapis.com/youtube/v3/playlistItems",
        params=params,
        timeout=30
    )
    res.raise_for_status()

    data = res.json()

    for item in data.get("items", []):
        snippet = item["snippet"]

        video_id = snippet["resourceId"]["videoId"]

        all_videos.append({
            "channel_id": channel_id,
            "channel_name": channel_name,
            "video_id": video_id,
            "title": snippet.get("title", ""),
            "description": snippet.get("description", ""),
            "published_at": snippet.get("publishedAt", ""),
            "url": f"https://www.youtube.com/watch?v={video_id}",
        })

    page_count += 1
    print(f"{page_count}페이지 수집 완료 / 누적 {len(all_videos)}개")

    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        break

    if MAX_PAGES is not None and page_count >= MAX_PAGES:
        break

    time.sleep(SLEEP_SECONDS)

df_videos = pd.DataFrame(all_videos)
df_videos.head()
df_videos.to_excel(OUTPUT_PATH, index=False)

1페이지 수집 완료 / 누적 50개
2페이지 수집 완료 / 누적 100개
3페이지 수집 완료 / 누적 150개
4페이지 수집 완료 / 누적 200개
5페이지 수집 완료 / 누적 250개
6페이지 수집 완료 / 누적 300개
7페이지 수집 완료 / 누적 350개
8페이지 수집 완료 / 누적 400개
9페이지 수집 완료 / 누적 450개
10페이지 수집 완료 / 누적 500개
11페이지 수집 완료 / 누적 550개
12페이지 수집 완료 / 누적 600개
13페이지 수집 완료 / 누적 650개
14페이지 수집 완료 / 누적 700개
15페이지 수집 완료 / 누적 750개
16페이지 수집 완료 / 누적 800개
17페이지 수집 완료 / 누적 850개
18페이지 수집 완료 / 누적 900개
19페이지 수집 완료 / 누적 950개
20페이지 수집 완료 / 누적 1000개
21페이지 수집 완료 / 누적 1050개
22페이지 수집 완료 / 누적 1100개
23페이지 수집 완료 / 누적 1150개
24페이지 수집 완료 / 누적 1200개
25페이지 수집 완료 / 누적 1250개
26페이지 수집 완료 / 누적 1300개
27페이지 수집 완료 / 누적 1350개
28페이지 수집 완료 / 누적 1400개
29페이지 수집 완료 / 누적 1450개
30페이지 수집 완료 / 누적 1500개
31페이지 수집 완료 / 누적 1550개
32페이지 수집 완료 / 누적 1600개
33페이지 수집 완료 / 누적 1650개
34페이지 수집 완료 / 누적 1700개
35페이지 수집 완료 / 누적 1719개


,channel_id,channel_name,video_id,title,description,published_at,url
0,UCaehHbVEHmss6hOYt-mjnWg,UN Trade and Development,XfNOBadZCZs,UNCTAD DSG Pedro Manuel Moreno at the WTO's 14...,Services are the fastest-growing part of globa...,2026-03-27T09:21:50Z,https://www.youtube.com/watch?v=XfNOBadZCZs
1,UCaehHbVEHmss6hOYt-mjnWg,UN Trade and Development,79s-wrgAw8I,Strait of Hormuz Disruptions - Portuguese Version,Disruptions in the Strait of Hormuz — one of t...,2026-03-18T17:06:32Z,https://www.youtube.com/watch?v=79s-wrgAw8I
2,UCaehHbVEHmss6hOYt-mjnWg,UN Trade and Development,y22VRDCincQ,Strait of Hormuz Disruptions - Hindi Version,Disruptions in the Strait of Hormuz — one of t...,2026-03-18T16:55:50Z,https://www.youtube.com/watch?v=y22VRDCincQ
3,UCaehHbVEHmss6hOYt-mjnWg,UN Trade and Development,_PLP7OvxlNw,Strait of Hormuz Disruptions - Urdu Version,Disruptions in the Strait of Hormuz — one of t...,2026-03-18T16:43:11Z,https://www.youtube.com/watch?v=_PLP7OvxlNw
4,UCaehHbVEHmss6hOYt-mjnWg,UN Trade and Development,u5MguyBWUZI,Strait of Hormuz Disruptions - Swahili Version,Disruptions in the Strait of Hormuz — one of t...,2026-03-18T16:40:28Z,https://www.youtube.com/watch?v=u5MguyBWUZI
